# Pràctica: Predir èxits a Spotify
**Alumne:** Raul Garcia Magro

### Quin és l'objectiu?
He triat un dataset de cançons de Spotify. L'objectiu és que els models veguin les característiques d'una cançó i endevinin si serà un **Èxit (1)** o un **Fracàs (0)**. Considero que és un èxit si la seva popularitat és major a 50.

In [36]:
import kagglehub
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# 1. Descarreguem les dades de Spotify
path = kagglehub.dataset_download("yashdev01/spotify-tracks-dataset")
fitxers = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, fitxers[0]))

# Portem les files que tinguin errors o estiguin buides
df = df.dropna()

### Preparar les dades
Abans de donar les dades als models, hem de fer dos passos importants:

1. **Triar les característiques**: Agafem només les columnes que descriuen el so d'una cançó (ritme, energia, volum...)

2. **Escalar les dades**: Igualem les dades. Per exemple, el *tempo* pot valer 150 i la *ballabilitat* 0.8. Si no ho igualem, l'ordinador pensarà que el tempo és molt més important, pero no. Igualant'lo, tots els valors queden en un rang similar i el model pot aprendre bé de tots ells.

In [37]:
# 2. Fem l'objectiu: 1 si és popular (>50), 0 si no ho és
df['is_popular'] = (df['popularity'] > 50).astype(int)

# 3. Ens quedem només amb les dades que descriuen el so
caracteristiques = ['danceability', 'energy', 'loudness', 'speechiness', 
                    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[caracteristiques]
y = df['is_popular']

# 4. Separem: 80% de cançons per estudiar i 20% per examinar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Escalem les dades perquè totes tinguin el mateix pes
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Model 1: k-Nearest Neighbors (k-NN)
**Com funciona?** És la regla dels veïns. Si li passo una cançó nova, buscarà les 5 cançons més semblants que ja coneix. Si la majoria d'aquestes 5 van ser èxits, dirà que la nova també ho serà.

In [38]:
# Entrenem el model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Resultat
pred_knn = knn.predict(X_test_scaled)
print(f"Nota del k-NN: {accuracy_score(y_test, pred_knn) * 100:.2f}% son exits")

Nota del k-NN: 75.55% son exits


### Model 2: Perceptron
**Com funciona?** Dibuixa una línia recta i les dades que estan per sobre la linea son exits i els que no, son fracàs.

In [39]:
# Entrenem el Perceptron
perceptron = Perceptron(max_iter=1000, random_state=42)
perceptron.fit(X_train_scaled, y_train)

# Resultat
pred_perc = perceptron.predict(X_test_scaled)
print(f"Nota del Perceptron: {accuracy_score(y_test, pred_perc) * 100:.2f}% son les cançons populars")

Nota del Perceptron: 68.52% son les cançons populars


### Model 3: Naive Bayes
**Com funciona?** Mira les característiques de la cançó de manera separada i calcula la probabilitat final de que sigui popular.

In [40]:
# Entrenem el Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

# Resultat
pred_nb = nb.predict(X_test_scaled)
print(f"Nota del Naive Bayes: {accuracy_score(y_test, pred_nb) * 100:.2f}% es popular")

Nota del Naive Bayes: 70.81% es popular


### Conclusions
1. El **k-NN** és el model que té més sentit per aquest dataset. Agrupa cançons semblants, que és precisament el que fa l'algoritme real de Spotify per recomanar-nos música.
2. El **Perceptron** No es el millor perquè no es pot trobar una fórmula lineal perfecta per crear un èxit musical.
3. El **Naive Bayes** No es bo ja que la popularitat d'una cançó depèn molt del màrqueting i la fama de l'artista, dades que aquests models no estan mirant.